# Clase 7 — Accesibilidad y cambio de unidad de análisis

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 7 — Análisis de accesibilidad |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Cuánta gente, y no cuánta superficie, queda lejos de un centro de salud?

La Clase 6 terminó con dos resultados y dos deudas.

Los resultados: a 500 metros de un efector público de salud está el **24 % de la superficie de
Recoleta** y el **49 % de la de Villa Lugano**, y la distancia mediana de un equipamiento al
efector más cercano es de 604 y 422 metros.

Las deudas. La primera es que esas distancias son **en línea recta**, y nadie camina en línea
recta. La segunda, y más seria, es que toda la cobertura se midió sobre **superficie**: el
territorio cubierto, no la gente que vive en él. Un barrio puede tener la mitad de su superficie
cubierta y a casi toda su población adentro, o al revés.

Hoy saldamos las dos. Para la segunda hace falta un dato que no teníamos: **población a escala
fina**, que es lo que aportan los radios censales.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Calcular** distancias y tiempos de viaje por la red de calles, y contrastarlos con la
   distancia en línea recta.
2. **Construir** isocronas y explicar en qué se diferencian de un área de influencia.
3. **Transferir** una variable de una división territorial a otra por pertenencia y por
   ponderación de área, y **decidir** cuál corresponde en cada caso.
4. **Recalcular** una cobertura sobre población en lugar de sobre superficie.
5. **Normalizar** variables en unidades distintas y **componer** un índice territorial.

## 3. Material de esta clase

| Archivo | Contenido | Fuente |
|---|---|---|
| `osm_barrios_limites.gpkg` | Contorno de Recoleta y Villa Lugano | OpenStreetMap (Clase 3) |
| `salud_barrios.gpkg` | Los 13 efectores públicos de salud de esos barrios | IGN (Clase 3) |
| `caba_radios_2022.gpkg` | 3.554 radios censales con población, hogares y hogares con NBI | BA Data — Censo 2022 |

Los dos primeros vienen de la Clase 6. El tercero es la fuente nueva, y es también la capa sobre
la que trabaja la Clase 8.

**Servicio externo.** El cálculo de rutas usa **OpenRouteService**, que requiere una clave
gratuita. La notebook trae una clave de cátedra para que funcione en clase; más abajo está el
enlace para obtener la propia, que es lo que conviene hacer para el trabajo final.

---

## 4. Medir la separación, y cambiar de unidad

### 4.1 Tres maneras de medir cuán lejos está algo

**Conceptos clave.** La palabra "distancia" esconde tres medidas distintas, que responden
preguntas distintas y pueden diferir mucho entre sí.

| Medida | Qué mide | Qué supone |
|---|---|---|
| **Distancia euclídea** | La separación en línea recta | Que el espacio es homogéneo y se puede atravesar en cualquier dirección |
| **Distancia por la red** | El recorrido más corto por las calles | Que existe una red, y que el desplazamiento se hace sobre ella |
| **Tiempo de viaje** | Cuánto se tarda | Todo lo anterior, más una velocidad y un modo de transporte |

La euclídea es la que usamos en la Clase 6 y tiene dos virtudes: no necesita más datos que las
coordenadas, y se calcula instantáneamente. Su defecto es que **siempre subestima**: ignora que
hay manzanas, vías del ferrocarril, autopistas sin cruce y ríos.

La razón entre la distancia por red y la euclídea se llama **índice de rodeo** o *detour index*.
En una trama amanzanada regular ronda 1,2 a 1,3; donde hay barreras —una autopista, un arroyo
entubado, un predio ferroviario— puede superar 2. Ese número no es un detalle técnico: es la
diferencia entre un servicio que está cerca y uno que está del otro lado de las vías.

### 4.2 Qué es una isocrona

Una **isocrona** es el polígono que delimita todo lo que se puede alcanzar desde un punto en un
tiempo dado, desplazándose por la red.

La diferencia con un área de influencia es de fondo. El **buffer** de la Clase 6 es una figura
geométrica: el conjunto de puntos a menos de 500 metros en línea recta, y por eso es siempre un
círculo. La **isocrona** es el resultado de recorrer la red: tiene forma de estrella o de
mancha irregular, se estira a lo largo de las avenidas y se corta donde hay barreras.

El buffer describe el espacio; la isocrona describe **el espacio tal como se lo puede recorrer**.
Por eso un buffer siempre contiene a la isocrona del mismo radio equivalente: promete un alcance
que la red no entrega.

### 4.3 El problema de la unidad de análisis

**Conceptos clave.** Rara vez la unidad en que viene un dato coincide con la unidad en que se
necesita. El censo publica por radio censal, la salud por área programática, la educación por
distrito escolar, la policía por comisaría, y el estudio se hace por barrio. Ninguna de esas
divisiones coincide con las otras.

Transferir una variable de una división a otra se llama **interpolación areal**, y hay dos
métodos básicos:

**Por pertenencia.** Cada unidad de origen se asigna entera a la unidad de destino que la
contiene —o en la que cae su punto interior—. Es simple y rápido. Funciona bien cuando las dos
divisiones están **anidadas**: los radios dentro de las fracciones, las fracciones dentro de las
comunas.

**Por ponderación de área.** Cada unidad de origen se reparte entre las de destino en proporción
a la superficie que comparten. Si un radio con 1.000 habitantes queda 40 % dentro de un barrio,
aporta 400.

El segundo método hace un supuesto fuerte que conviene tener presente: **que la variable está
distribuida de manera uniforme dentro de la unidad de origen**. Para la superficie eso es
trivialmente cierto; para la población no lo es —dentro de un radio hay manzanas edificadas y
plazas vacías— y ahí el método introduce un error que no se puede cuantificar con estos datos.

Es la misma cuestión que la Clase 2 planteó como **problema de la unidad de área modificable**
(MAUP). Allá se mostró que el resultado cambia con la grilla; acá se mide cuánto cambia y se
decide qué hacer.

---

## 5. Preparación

### ▶️ Bibliotecas

`openrouteservice` es nueva: es el cliente del servicio de ruteo.

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "matplotlib==3.9.2" \
               "folium==0.17.0" "openrouteservice==2.3.3"

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import openrouteservice

print("Bibliotecas listas")

### ▶️ Las capas de la Clase 6

Los dos barrios y los 13 efectores de salud, con las versiones métricas para medir.

In [ ]:
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

GEOGRAFICO = "EPSG:4326"
METRICO    = "EPSG:5347"

barrios = gpd.read_file(DATOS + "osm_barrios_limites.gpkg")
salud   = gpd.read_file(DATOS + "salud_barrios.gpkg")

barrios_m, salud_m = barrios.to_crs(METRICO), salud.to_crs(METRICO)

print(f"{len(barrios)} barrios · {len(salud)} efectores de salud")

---

## 6. Accesibilidad por la red de calles

**Escenario.** Tomamos un punto representativo de cada barrio y medimos cuánto hay hasta su
efector de salud más cercano de dos maneras: en línea recta, como en la Clase 6, y caminando por
las calles. Después construimos la isocrona de diez minutos a pie alrededor de un efector y la
comparamos con el buffer de 500 metros.

El servicio es **OpenRouteService**, que calcula rutas e isocronas sobre la red de
OpenStreetMap. Es gratuito y su cuota alcanza de sobra para una clase.

### ▶️ La clave del servicio

La celda siguiente trae una clave de cátedra, para que la notebook funcione durante el
encuentro. **Para el trabajo final conviene usar una propia**, que se obtiene en un par de
minutos y sin costo:

1. Crear una cuenta en [openrouteservice.org/dev/#/signup](https://openrouteservice.org/dev/#/signup).
2. En el panel, pedir un token del plan gratuito.
3. Pegarlo abajo, reemplazando el valor de `ORS_API_KEY`.

La cuota gratuita admite 2.000 pedidos por día y 40 por minuto, muy por encima de lo que
necesita esta clase.

In [ ]:
# Reemplazá esta clave por la tuya: openrouteservice.org/dev/#/signup
ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjcyYTU2OTgyYzRkMzg3MGEzNDVmMGI3YWE4MzIyMWViZWNiNjVmMDkwZjI2ZWIwZjQ1MjNjMmU4IiwiaCI6Im11cm11cjY0In0="

cliente = openrouteservice.Client(key=ORS_API_KEY)
print("Cliente de OpenRouteService listo")

### ▶️ De dónde a dónde

Un punto interior por barrio —`representative_point()`, como en la Clase 6— y, para cada uno, el
efector de salud más cercano en línea recta.

In [ ]:
origenes_m = barrios_m.copy()
origenes_m["geometry"] = barrios_m.representative_point()

cercano_m = gpd.sjoin_nearest(
    origenes_m, salud_m[["nombre", "geometry"]],
    distance_col="dist_recta_m").drop(columns="index_right")

cercano_m[["barrio", "nombre", "dist_recta_m"]].round(0)

### ▶️ La ruta a pie

`directions` devuelve el recorrido y un resumen con la distancia y la duración. Las coordenadas
van en EPSG:4326 y en el orden **longitud, latitud**.

In [ ]:
origenes = origenes_m.to_crs(GEOGRAFICO)
destinos = salud.set_index("nombre")

rutas = []
for (_, origen), (_, fila) in zip(origenes.iterrows(), cercano_m.iterrows()):
    destino = destinos.loc[fila["nombre"]].geometry
    respuesta = cliente.directions(
        coordinates=[[origen.geometry.x, origen.geometry.y], [destino.x, destino.y]],
        profile="foot-walking", format="geojson")
    resumen = respuesta["features"][0]["properties"]["summary"]
    rutas.append({"barrio": origen["barrio"], "efector": fila["nombre"],
                  "dist_recta_m": round(fila["dist_recta_m"]),
                  "dist_red_m": round(resumen["distance"]),
                  "minutos": round(resumen["duration"] / 60, 1),
                  "geometry": respuesta["features"][0]["geometry"]})

rutas_gdf = gpd.GeoDataFrame(
    rutas, geometry=gpd.GeoSeries.from_xy([0] * len(rutas), [0] * len(rutas)), crs=GEOGRAFICO)
print(f"{len(rutas)} rutas calculadas")

In [ ]:
from shapely.geometry import shape

rutas_gdf["geometry"] = [shape(r["geometry"]) for r in rutas]
rutas_gdf["rodeo"] = (rutas_gdf["dist_red_m"] / rutas_gdf["dist_recta_m"]).round(2)

rutas_gdf[["barrio", "efector", "dist_recta_m", "dist_red_m", "minutos", "rodeo"]]

### 👀 La ruta sobre el mapa

El segmento recto entre origen y destino, y el recorrido que devolvió el servicio. La diferencia
entre los dos es lo que la distancia euclídea no ve.

In [ ]:
mapa = barrios.explore(color="grey", style_kwds=dict(fill=False, weight=3),
                       tiles="CartoDB positron", name="Barrios")

rutas_gdf.explore(m=mapa, color="#c44e52", style_kwds=dict(weight=4),
                  tooltip=["barrio", "dist_red_m", "minutos"], name="Ruta a pie")

origenes.explore(m=mapa, color="black", marker_kwds=dict(radius=5), name="Origen")
salud.explore(m=mapa, color="navy", marker_kwds=dict(radius=5),
              tooltip=["nombre"], name="Efectores")

folium.LayerControl().add_to(mapa)
mapa

### ▶️ Las isocronas de caminata

`isochrones` recibe un punto y una lista de rangos en segundos. Las pedimos para los 13
efectores, en tres rangos: 5, 10 y 15 minutos a pie. Son trece consultas con una pausa entre
ellas para respetar la cuota del servicio, así que la celda tarda cerca de un minuto.

In [ ]:
import time

MINUTOS = [5, 10, 15]
filas = []

for _, efector in salud.iterrows():
    respuesta = cliente.isochrones(
        locations=[[efector.geometry.x, efector.geometry.y]],
        profile="foot-walking", range=[m * 60 for m in MINUTOS], range_type="time")
    for rasgo in respuesta["features"]:
        filas.append({"nombre": efector["nombre"], "barrio": efector["barrio"],
                      "minutos": rasgo["properties"]["value"] / 60,
                      "geometry": shape(rasgo["geometry"])})
    time.sleep(3)                       # la cuota admite 20 pedidos por minuto

print(f"{len(filas)} isocronas recibidas")

In [ ]:
isocronas = gpd.GeoDataFrame(filas, crs=GEOGRAFICO)
isocronas_m = isocronas.to_crs(METRICO)
isocronas_m["superficie_ha"] = (isocronas_m.area / 10_000).round(1)

isocronas_m[["barrio", "nombre", "minutos", "superficie_ha"]].head()

### 👀 La forma de una isocrona

Las tres isocronas de un efector, sobre el buffer de 500 metros de la Clase 6. Caminando a paso
normal, 500 metros son unos **seis minutos**, de modo que la isocrona de 5 minutos es la
comparable.

In [ ]:
EFECTOR = "Hospital General de Niños Doctor Ricardo Gutiérrez"

elegido_m = salud_m[salud_m["nombre"] == EFECTOR]
tres_m = isocronas_m[isocronas_m["nombre"] == EFECTOR]
buffer_m = gpd.GeoDataFrame(geometry=[elegido_m.geometry.iloc[0].buffer(500)], crs=METRICO)

fig, eje = plt.subplots(figsize=(9, 9))
buffer_m.plot(ax=eje, facecolor="none", edgecolor="black", linewidth=2, linestyle="--")
for minutos, color in zip([15, 10, 5], ["#fee0d2", "#fc9272", "#c44e52"]):
    tres_m[tres_m["minutos"] == minutos].plot(ax=eje, facecolor=color, edgecolor="none")
elegido_m.plot(ax=eje, color="black", markersize=60)

eje.set_title("Isocronas de 15, 10 y 5 minutos a pie, y el buffer de 500 m (línea punteada)")
eje.set_axis_off()
plt.show()

La isocrona no es un círculo: se estira sobre las avenidas, se quiebra en las esquinas y deja
huecos donde la trama no permite pasar. Esa forma **es** la red de calles, dibujada por el tiempo
de caminata.

### ▶️ ¿Es representativo ese efector?

Un solo caso no alcanza para concluir nada: la forma y el tamaño de una isocrona dependen de la
trama que rodea a cada punto. Con los trece podemos ver la distribución.

In [ ]:
resumen_iso = isocronas_m.groupby("minutos")["superficie_ha"].agg(
    ["min", "median", "max"]).round(1)

AREA_BUFFER_HA = np.pi * 500 ** 2 / 10_000
print(f"Buffer de 500 m: {AREA_BUFFER_HA:.1f} ha")
resumen_iso

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))

for minutos, color in zip(MINUTOS, ["#c44e52", "#fc9272", "#fee0d2"]):
    datos = isocronas_m.loc[isocronas_m["minutos"] == minutos, "superficie_ha"]
    eje.scatter(datos, [minutos] * len(datos), color=color, s=60,
                edgecolor="grey", label=f"{minutos:.0f} min")

eje.axvline(AREA_BUFFER_HA, color="black", linestyle="--", linewidth=1.5)
eje.text(AREA_BUFFER_HA + 5, 14, "buffer de 500 m", fontsize=9)
eje.set_xlabel("Superficie alcanzable (ha)")
eje.set_ylabel("Minutos de caminata")
eje.set_yticks(MINUTOS)
eje.legend(title="")
plt.tight_layout()
plt.show()

### 🔍 Interpretación

El gráfico ordena los trece efectores según la superficie que alcanzan, y la línea punteada
marca las 78,5 hectáreas del círculo de 500 metros.

**Ninguna isocrona de 5 minutos llega a esa línea.** La mediana es de 33,2 hectáreas: menos de
la mitad. Dicho de otro modo, el área de influencia de la Clase 6 prometía un alcance que la red
de calles no entrega, porque contaba como cubierto el interior de manzanas sin acceso y el otro
lado de barreras que hay que rodear. **La cobertura que calculamos en la Clase 6 estaba
sobreestimada**, y ahora sabemos aproximadamente por cuánto.

La dispersión entre efectores también informa. A cinco minutos van de 11,5 a 37,8 hectáreas: más
del triple entre el peor y el mejor ubicado. Esa diferencia no depende del efector sino de la
trama que lo rodea, y es la razón por la que un umbral único —"500 metros"— trata como
equivalentes situaciones que no lo son.

El índice de rodeo de la tabla anterior dice lo mismo para las rutas: por cada metro en línea
recta hay que caminar entre 1,2 y 1,3 metros en esta trama, y más donde hay barreras.

> ⚠️ **Lo que la isocrona tampoco dice.** Supone que se camina, a una velocidad constante, y que
> todas las calles son transitables. No distingue una avenida de doce carriles sin semáforo de
> una calle interna, ni tiene en cuenta la pendiente, la iluminación o la sensación de
> seguridad. Es una mejora sobre el círculo, no una medida del acceso real.

---

## 7. La fuente que faltaba: los radios censales

**Escenario.** Para responder la pregunta de la clase hace falta población a una escala más fina
que el barrio. El **radio censal** es la unidad mínima que publica el censo: agrupa unas 300
viviendas y en una ciudad densa mide pocas manzanas.

La Dirección General de Estadística y Censos de la Ciudad publica los resultados del Censo 2022
por radio, con geometría. Son 3.554 radios para toda la ciudad.

### ▶️ La carga

La capa viene del portal ya curada para el curso: se conservaron el código de radio, la comuna,
la población, los hogares y los hogares con NBI.

In [ ]:
radios = gpd.read_file(DATOS + "caba_radios_2022.gpkg")
radios_m = radios.to_crs(METRICO)

print(f"{len(radios)} radios censales · {int(radios.poblacion.sum()):,} habitantes".replace(",", "."))
radios.head(3)

### ✅ Comprobación — los radios sin hogares

Antes de dividir conviene mirar los denominadores. Hay dos radios con cero hogares: son
superficies sin uso residencial —una terminal, un parque— que el censo incluye igual para que la
capa cubra la ciudad entera sin huecos.

Calcular el porcentaje de NBI sobre ellos daría una división por cero. Se los deja fuera del
cálculo de la variable, pero **no de la capa**: siguen ocupando su lugar en el territorio.

In [ ]:
sin_hogares = radios_m["hogares"] == 0
print(f"Radios sin hogares: {sin_hogares.sum()}")

radios_m["perc_nbi"] = np.where(
    sin_hogares, np.nan, radios_m["hogares_nbi"] / radios_m["hogares"] * 100)

radios_m[["codigo", "comuna", "poblacion", "hogares", "hogares_nbi", "perc_nbi"]].head()

### 👀 La ciudad entera

Dos mapas: dónde vive la gente y dónde está el NBI. La densidad se calcula sobre la superficie
del radio, que es una variable territorial más, construida como en la Clase 6.

In [ ]:
radios_m["superficie_km2"] = radios_m.area / 1e6
radios_m["densidad_hab_km2"] = radios_m["poblacion"] / radios_m["superficie_km2"]

fig, ejes = plt.subplots(1, 2, figsize=(15, 8))

radios_m.plot(column="densidad_hab_km2", cmap="Blues", scheme="FisherJenks", k=5,
              legend=True, edgecolor="none", ax=ejes[0],
              legend_kwds=dict(title="hab/km²", loc="upper left", fontsize=8))
ejes[0].set_title("Densidad de población")

radios_m.plot(column="perc_nbi", cmap="Reds", scheme="FisherJenks", k=5, legend=True,
              edgecolor="none", ax=ejes[1], missing_kwds=dict(color="lightgrey"),
              legend_kwds=dict(title="% hogares con NBI", loc="upper left", fontsize=8))
ejes[1].set_title("Hogares con NBI")

for eje in ejes:
    barrios_m.boundary.plot(ax=eje, color="black", linewidth=1.2)
    eje.set_axis_off()
plt.tight_layout()
plt.show()

Los dos barrios de la Clase 6 aparecen marcados en negro, cada uno en un extremo del patrón: el
NBI alto se concentra en el sur, y Villa Lugano está ahí.

---

## 8. Cambiar de unidad de análisis: de radios a barrios

**Escenario.** Los radios traen la población; los barrios son nuestra unidad de estudio. Hay que
pasar el dato de una división a la otra, y para eso están los dos métodos de la sección 4.3.

El primer paso es ver si las divisiones coinciden.

### ▶️ ¿Encajan los radios en los barrios?

In [ ]:
radios_m["area_radio_m2"] = radios_m.area
zona_estudio = barrios_m.geometry.union_all()
radios_zona_m = radios_m[radios_m.intersects(zona_estudio)].copy()

fraccion_en_zona = (radios_zona_m.geometry.intersection(zona_estudio).area
                    / radios_zona_m["area_radio_m2"])
completos = int((fraccion_en_zona > 0.99).sum())

print(f"Radios que tocan la zona de estudio : {len(radios_zona_m)}")
print(f"  íntegramente adentro             : {completos}")
print(f"  partidos por el límite           : {len(radios_zona_m) - completos}")

### ▶️ Método 1 — Por pertenencia

Cada radio se asigna entero al barrio donde cae su punto interior. Es un `sjoin` como el de la
Clase 6, seguido de un `groupby`.

In [ ]:
puntos_m = radios_zona_m.copy()
puntos_m["geometry"] = radios_zona_m.representative_point()

por_pertenencia = gpd.sjoin(puntos_m, barrios_m[["barrio", "geometry"]],
                            predicate="within").drop(columns="index_right")

pertenencia = por_pertenencia.groupby("barrio").agg(
    radios=("codigo", "size"), poblacion=("poblacion", "sum"),
    hogares=("hogares", "sum"), hogares_nbi=("hogares_nbi", "sum"))
pertenencia

### ▶️ Método 2 — Por ponderación de área

`overlay` con `intersection` corta los radios contra los barrios y devuelve un fragmento por cada
par. La fracción del radio que quedó en el fragmento es el peso con que se reparte la población.

In [ ]:
fragmentos_m = gpd.overlay(radios_zona_m, barrios_m[["barrio", "geometry"]], how="intersection")
fragmentos_m["fraccion"] = fragmentos_m.area / fragmentos_m["area_radio_m2"]

for columna in ["poblacion", "hogares", "hogares_nbi"]:
    fragmentos_m[columna + "_pond"] = fragmentos_m[columna] * fragmentos_m["fraccion"]

ponderacion = fragmentos_m.groupby("barrio").agg(
    fragmentos=("codigo", "size"), poblacion=("poblacion_pond", "sum"),
    hogares=("hogares_pond", "sum"), hogares_nbi=("hogares_nbi_pond", "sum")).round(0)
ponderacion

### 👀 Los fragmentos parciales

Los trozos de radio que el límite del barrio dejó a medias, coloreados por la fracción del radio
original que representan.

In [ ]:
parciales_m = fragmentos_m[fragmentos_m["fraccion"] < 0.99]

fig, eje = plt.subplots(figsize=(9, 8))
fragmentos_m.plot(ax=eje, facecolor="#f0f0f0", edgecolor="white", linewidth=0.3)
parciales_m.plot(ax=eje, column="fraccion", cmap="viridis", legend=True, edgecolor="none",
                 legend_kwds=dict(label="fracción del radio original", shrink=0.6))
barrios_m.boundary.plot(ax=eje, color="black", linewidth=1.5)

eje.set_title(f"{len(parciales_m)} fragmentos parciales sobre {len(fragmentos_m)}")
eje.set_axis_off()
plt.show()

### ▶️ Los dos métodos, comparados

In [ ]:
comparacion = pd.DataFrame({
    "por_pertenencia": pertenencia["poblacion"],
    "por_ponderacion": ponderacion["poblacion"],
})
comparacion["diferencia"] = comparacion["por_pertenencia"] - comparacion["por_ponderacion"]
comparacion["diferencia_perc"] = (comparacion["diferencia"]
                                  / comparacion["por_ponderacion"] * 100).round(2)
comparacion

In [ ]:
poblacion_en_juego = (parciales_m["poblacion"] * parciales_m["fraccion"]).sum()
total = fragmentos_m["poblacion_pond"].sum()

print(f"Fracción de los fragmentos parciales — mediana: {parciales_m['fraccion'].median():.3f}")
print(f"Población que vive en ellos: {poblacion_en_juego:,.0f} de {total:,.0f} "
      f"({100 * poblacion_en_juego / total:.1f} %)".replace(",", "."))

### 🔍 Interpretación — cuándo hace falta ponderar

Los dos métodos dan **prácticamente el mismo resultado**: la diferencia no llega al 0,2 % en
ninguno de los dos barrios. Vale la pena entender por qué, porque la explicación es la regla de
uso.

Los fragmentos parciales son 82 sobre 381, pero la mediana de su fracción es **0,006**: son
astillas. El límite de los barrios de OpenStreetMap sigue las calles, y los radios censales
también, de modo que las dos divisiones están casi alineadas y lo que queda a medias son restos
de un par de metros a lo largo del eje de la calzada. Sólo el **2,4 % de la población** está en
esos fragmentos, y los errores por exceso y por defecto se compensan entre sí.

**La regla:** cuando las divisiones están anidadas o siguen la misma trama, la asignación por
pertenencia alcanza, y es mucho más barata de calcular. La ponderación por área se vuelve
necesaria cuando la geometría de destino **no respeta** la división de origen —un área
programática de salud, un radio de influencia, una cuenca, un polígono de riesgo—, y ese es
exactamente el caso del bloque siguiente.

---

## 9. La cobertura, ahora sobre población

**Escenario.** Volvemos a la pregunta de la Clase 6, con el dato que faltaba. La zona de
cobertura se construye igual —buffers de 500 metros disueltos— pero ahora lo que se mide no es
qué parte del territorio queda adentro sino **qué parte de la gente**.

Acá la ponderación por área deja de ser una alternativa y pasa a ser obligatoria: la zona de
cobertura es un conjunto de círculos que no tiene ninguna relación con los límites censales, y
parte los radios por el medio.

In [ ]:
RADIO_M = 500
cobertura = salud_m.buffer(RADIO_M).union_all()

radios_zona_m["fraccion_cubierta"] = (
    radios_zona_m.geometry.intersection(cobertura).area / radios_zona_m["area_radio_m2"])
radios_zona_m["punto_cubierto"] = radios_zona_m.representative_point().within(cobertura)

partidos_m = radios_zona_m[radios_zona_m["fraccion_cubierta"].between(0.15, 0.85)]
print(f"Radios partidos por la zona de cobertura: {len(partidos_m)}")
print(f"Población que vive en ellos: {int(partidos_m.poblacion.sum()):,}".replace(",", "."))

### 👀 Por qué acá la pertenencia no sirve

La tabla muestra los radios más poblados de los que quedaron partidos. La columna
`cuenta_pertenencia` es la población que el método simple les atribuiría —todo o nada, según
dónde caiga el punto interior— y `cuenta_ponderacion`, la que corresponde a la fracción
efectivamente cubierta.

In [ ]:
ejemplos = partidos_m.nlargest(6, "poblacion")[
    ["codigo", "poblacion", "fraccion_cubierta", "punto_cubierto"]].copy()

ejemplos["cubierto_perc"] = (ejemplos["fraccion_cubierta"] * 100).round(0)
ejemplos["cuenta_pertenencia"] = np.where(ejemplos["punto_cubierto"], ejemplos["poblacion"], 0)
ejemplos["cuenta_ponderacion"] = (ejemplos["poblacion"] * ejemplos["fraccion_cubierta"]).round(0)

ejemplos[["codigo", "poblacion", "cubierto_perc", "cuenta_pertenencia", "cuenta_ponderacion"]]

El primero de la lista está cubierto en un **58 %** y sin embargo la pertenencia le asigna
**cero**, porque su punto interior cayó justo afuera. El tercero está cubierto en un 51 % y la
pertenencia le cuenta **toda** su población. En un solo radio el error es de miles de personas,
y hay 51 radios en esa situación.

En el total esos errores tienden a compensarse, pero **la compensación es una casualidad, no un
método**. Con la ponderación cada radio aporta lo que le corresponde.

### ▶️ El cálculo

Se combinan las dos operaciones: primero repartir cada radio entre los barrios, después medir
qué fracción de cada fragmento cae dentro de la zona de cobertura.

In [ ]:
fragmentos_m["fraccion_cubierta"] = (
    fragmentos_m.geometry.intersection(cobertura).area / fragmentos_m.area)
fragmentos_m["poblacion_cubierta"] = (
    fragmentos_m["poblacion_pond"] * fragmentos_m["fraccion_cubierta"])

resultado = fragmentos_m.groupby("barrio").agg(
    poblacion=("poblacion_pond", "sum"), poblacion_cubierta=("poblacion_cubierta", "sum"))
resultado["cobertura_poblacion_perc"] = (
    resultado["poblacion_cubierta"] / resultado["poblacion"] * 100).round(1)

resultado.round(0)

### 👀 El contraste con la Clase 6

La cobertura sobre superficie se recalcula acá para tenerla al lado, y tiene que dar lo mismo que
la clase pasada: 23,7 % en Recoleta y 49,2 % en Villa Lugano.

In [ ]:
resultado["cobertura_superficie_perc"] = [
    round(100 * b.intersection(cobertura).area / b.area, 1) for b in barrios_m.geometry]

comparar = resultado[["cobertura_superficie_perc", "cobertura_poblacion_perc"]]

eje = comparar.plot.barh(figsize=(9, 4), color=["#bdbdbd", "#c44e52"], width=0.7)
eje.set_xlabel("% cubierto a menos de 500 m de un efector")
eje.set_ylabel("")
eje.legend(["Sobre superficie (Clase 6)", "Sobre población"])
eje.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

### ▶️ Por qué la población está mejor cubierta que el territorio

La diferencia entre las dos barras se explica con una cuenta: la densidad de población adentro y
afuera de la zona cubierta.

In [ ]:
for barrio in barrios_m["barrio"]:
    grupo = fragmentos_m[fragmentos_m["barrio"] == barrio]
    area_cub = grupo.geometry.intersection(cobertura).area.sum() / 1e6
    area_fuera = grupo.area.sum() / 1e6 - area_cub
    pob_cub = grupo["poblacion_cubierta"].sum()
    pob_fuera = grupo["poblacion_pond"].sum() - pob_cub
    print(f"{barrio:14} dentro: {pob_cub / area_cub:8,.0f} hab/km²   "
          f"fuera: {pob_fuera / area_fuera:8,.0f} hab/km²".replace(",", "."))

### 🔍 Interpretación

**Recoleta pasa del 24 % de superficie cubierta al 43 % de población cubierta. Villa Lugano, del
49 % al 68 %.** En los dos barrios la cobertura sobre población es sustancialmente mayor, y la
razón es la misma: **dentro de la zona de cobertura vive más del doble de gente por kilómetro
cuadrado que fuera de ella**.

Eso no es casualidad ni un artefacto del método. Los efectores de salud están donde está la
gente: en el tejido denso del barrio y no en los predios ferroviarios, los parques o las zonas
industriales, que ocupan superficie y no tienen población. Medir cobertura sobre superficie los
cuenta como territorio descubierto, y no hay nadie ahí a quien cubrir.

**La conclusión sustantiva de la Clase 6 se sostiene y se refuerza:** Villa Lugano sigue mejor
cubierta que Recoleta, ahora por 68 % contra 43 %. Lo que cambia es la magnitud del problema: el
57 % de la población de Recoleta —unas 90.000 personas— vive a más de 500 metros de un efector
público, y en Villa Lugano son unas 40.000.

Esa es la cifra que va en un informe. "El 24 % de la superficie está cubierta" no le dice nada a
nadie; "90.000 personas viven a más de seis minutos a pie de un centro de salud público" sí.

---

## 10. Normalizar y componer un índice

**Conceptos clave.** La tabla que venimos armando tiene columnas en unidades incompatibles:
porcentajes, metros, habitantes por kilómetro cuadrado. No se pueden sumar ni promediar tal como
están, y cualquier comparación entre ellas está dominada por la que tenga los números más
grandes.

**Normalizar** es llevarlas todas a una escala común. Los dos procedimientos habituales son:

**Min–max.** Se reescala al intervalo 0–1 restando el mínimo y dividiendo por el rango:

$$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$

Es intuitivo —0 es el peor caso observado, 1 el mejor— pero depende de los extremos: un solo
valor atípico comprime a todos los demás.

**Z-score.** Se expresa cada valor en desvíos estándar respecto de la media:

$$z = \frac{x - \bar{x}}{s}$$

No tiene límites fijos, es robusto a los extremos, y tiene la ventaja de que el signo dice de
qué lado de la media está cada unidad.

Un **índice compuesto** combina varias variables normalizadas en un solo número, casi siempre
por suma ponderada. Es una herramienta útil y también la operación más fácil de usar mal: los
pesos son una decisión del analista y no salen de los datos.

### ▶️ Las variables del índice

Construimos un índice de **desventaja territorial** sobre los radios de la zona de estudio, con
tres componentes: el NBI, la falta de cobertura de salud y la distancia al efector más cercano.
Los tres apuntan en el mismo sentido: más alto, peor.

In [ ]:
indice_m = radios_zona_m[radios_zona_m["hogares"] > 0].copy()

indice_m["sin_cobertura"] = (1 - indice_m["fraccion_cubierta"]) * 100

centros_m = gpd.GeoDataFrame(geometry=indice_m.representative_point(), crs=METRICO)
cercano = gpd.sjoin_nearest(centros_m, salud_m[["geometry"]], distance_col="dist_salud_m")
indice_m["dist_salud_m"] = cercano.groupby(level=0)["dist_salud_m"].min()

COMPONENTES = ["perc_nbi", "sin_cobertura", "dist_salud_m"]
indice_m[COMPONENTES].describe().round(1)

### ▶️ Las dos normalizaciones

In [ ]:
for componente in COMPONENTES:
    valores = indice_m[componente]
    indice_m[componente + "_mm"] = (valores - valores.min()) / (valores.max() - valores.min())
    indice_m[componente + "_z"] = (valores - valores.mean()) / valores.std()

indice_m[[c + "_mm" for c in COMPONENTES] + [c + "_z" for c in COMPONENTES]].describe().round(2)

El min–max deja todas las columnas entre 0 y 1, pero con medias muy distintas: donde hay valores
atípicos, la mayoría de las unidades queda apretada cerca de 0. El z-score tiene media 0 y desvío
1 en las tres, que es lo que permite sumarlas sin que una domine.

### ▶️ El índice

Con pesos iguales, que es el supuesto más transparente: las tres dimensiones valen lo mismo.

In [ ]:
PESOS = {"perc_nbi": 1 / 3, "sin_cobertura": 1 / 3, "dist_salud_m": 1 / 3}

indice_m["indice_desventaja"] = sum(
    indice_m[componente + "_z"] * peso for componente, peso in PESOS.items())

indice_m[["codigo", "poblacion"] + COMPONENTES + ["indice_desventaja"]].nlargest(
    5, "indice_desventaja").round(1)

In [ ]:
fig, eje = plt.subplots(figsize=(9, 8))

indice_m.plot(column="indice_desventaja", cmap="RdYlBu_r", scheme="FisherJenks", k=5,
              legend=True, edgecolor="white", linewidth=0.2, ax=eje,
              legend_kwds=dict(title="Índice de desventaja", loc="upper left", fontsize=8))
barrios_m.boundary.plot(ax=eje, color="black", linewidth=1.5)
salud_m.plot(ax=eje, color="black", marker="P", markersize=60)

eje.set_title("Índice compuesto de desventaja territorial, por radio censal")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación, y una advertencia

El mapa identifica los radios que combinan las tres desventajas, y esa es la utilidad del índice:
un solo número que ordena las unidades para priorizar.

Pero conviene ser explícito sobre lo que el índice esconde, porque es mucho.

**Los pesos son una decisión.** Con un tercio para cada dimensión estamos afirmando que estar
lejos de un centro de salud pesa lo mismo que tener necesidades básicas insatisfechas. Nada en
los datos justifica esa equivalencia; con otros pesos, el orden cambia. Si el índice va a usarse
para priorizar, los pesos se discuten y se declaran.

**Las componentes no son independientes.** La distancia al efector y la falta de cobertura miden
casi lo mismo desde dos ángulos, así que esa dimensión entra con doble peso sin que nadie lo haya
decidido. Antes de componer un índice conviene mirar la correlación entre sus partes.

**Y el resultado ya no tiene unidad.** Un valor de 1,4 no significa nada por sí solo: solo sirve
para comparar radios entre sí, dentro de esta ciudad y con estos datos.

In [ ]:
indice_m[COMPONENTES].corr().round(2)

---

## 11. La tabla de la clase

### ▶️ Composición y exportación

In [ ]:
tabla = resultado.reset_index()[
    ["barrio", "poblacion", "cobertura_superficie_perc", "cobertura_poblacion_perc"]].copy()

tabla["poblacion"] = tabla["poblacion"].round(0)
tabla["poblacion_sin_cobertura"] = (
    tabla["poblacion"] * (1 - tabla["cobertura_poblacion_perc"] / 100)).round(0)
tabla["radios"] = tabla["barrio"].map(pertenencia["radios"])

tabla

In [ ]:
salida = barrios[["barrio", "geometry"]].merge(tabla, on="barrio")
salida.to_file("accesibilidad_barrios.gpkg", driver="GPKG")
tabla.to_csv("accesibilidad_barrios.csv", index=False)

indice_m[["codigo", "comuna", "poblacion"] + COMPONENTES + ["indice_desventaja", "geometry"]] \
    .to_file("indice_desventaja_radios.gpkg", driver="GPKG")

print("accesibilidad_barrios.gpkg / .csv  — la tabla por barrio")
print("indice_desventaja_radios.gpkg      — el índice por radio censal")

---

## 12. Cierre

### El recorrido de hoy

| Operación | Pregunta que responde | Resultado |
|---|---|---|
| Ruteo por red | ¿Cuánto hay que caminar de verdad? | Índice de rodeo por encima de 1 en las dos rutas |
| Isocrona | ¿Hasta dónde se llega en 5 minutos? | Bastante menos que el círculo de 500 m |
| Pertenencia | ¿Qué radios son de cada barrio? | 198 y 108 radios |
| Ponderación por área | ¿Cuánta población le toca a cada barrio? | Igual que por pertenencia: las divisiones están alineadas |
| Cobertura sobre población | ¿Cuánta gente está cerca? | 43 % en Recoleta, 68 % en Villa Lugano |
| Normalización e índice | ¿Cómo combino variables incomparables? | Índice de desventaja por radio |

### La idea para llevarse

Las dos deudas de la Clase 6 se saldaron con el mismo movimiento: **cambiar la unidad en la que
se mide**.

La primera fue cambiar de la línea recta a la red. La segunda, de la superficie a la población. Y
en los dos casos el resultado cambió de magnitud sin cambiar de sentido: la conclusión de la
Clase 6 —Villa Lugano mejor cubierta que Recoleta— se sostuvo, pero las cifras que ahora podemos
informar son otras, y son las que le importan a alguien que tenga que decidir algo.

Que una conclusión sobreviva a un cambio de método es la mejor evidencia de que no era un
artefacto. Que las cifras cambien es el recordatorio de que **la unidad de medida es parte del
resultado**.

### Glosario

| Término | Definición |
|---|---|
| **Distancia por la red** | Longitud del recorrido más corto sobre una red de circulación. |
| **Índice de rodeo** | Razón entre la distancia por la red y la distancia en línea recta. |
| **Isocrona** | Polígono que delimita lo alcanzable desde un punto en un tiempo dado, recorriendo la red. |
| **Interpolación areal** | Transferencia de una variable de una división territorial a otra. |
| **Asignación por pertenencia** | Cada unidad de origen se asigna entera a la unidad de destino que la contiene. |
| **Ponderación por área** | Cada unidad de origen se reparte en proporción a la superficie compartida. Supone distribución uniforme. |
| **Radio censal** | Unidad mínima de publicación del censo; agrupa unas 300 viviendas. |
| **Normalización min–max** | Reescalado al intervalo 0–1. Sensible a los valores extremos. |
| **Z-score** | Expresión en desvíos estándar respecto de la media. |
| **Índice compuesto** | Combinación ponderada de variables normalizadas. Los pesos son una decisión, no un dato. |

### Tres preguntas para autoevaluarse

1. Tenés datos por radio censal y necesitás totales por comuna. ¿Qué método usarías y por qué?
2. Un buffer de 1 km y una isocrona de 12 minutos a pie cubren áreas parecidas. ¿Son
   intercambiables? ¿Qué le preguntarías a cada uno?
3. Un índice compuesto de tres variables ordena los barrios de tu estudio. Un colega cambia los
   pesos y el orden se invierte. ¿Qué hay que revisar antes de publicar cualquiera de los dos?

### Qué viene en la Clase 8

Hoy construimos una variable por radio censal para toda la ciudad. La próxima clase la
interroga: **¿ese patrón que se ve en el mapa es real, o es lo que el ojo arma con cualquier
mancha de color?**

Vamos a medir si los radios se parecen a sus vecinos más de lo que se parecerían por azar, a
localizar dónde están los conglomerados, y a decir hasta qué distancia llega esa semejanza.

---

## 13. Referencias

- Goodchild, M. F. y Lam, N. S.-N. (1980). "Areal interpolation: a variant of the traditional
  spatial problem". *Geo-Processing*, 1, 297–312.
- Openshaw, S. (1984). *The Modifiable Areal Unit Problem*. CATMOG 38.
- Rey, S., Arribas-Bel, D. y Wolf, L. J. (2023). *Geographic Data Science with Python*, cap. 9
  "Spatial Inequality". CRC Press.
- OECD y Joint Research Centre (2008). *Handbook on Constructing Composite Indicators:
  Methodology and User Guide*. OECD Publishing.
- Dirección General de Estadística y Censos del Gobierno de la Ciudad de Buenos Aires (2023).
  *Censo Nacional de Población, Hogares y Viviendas 2022. Resultados por radio censal*.
- OpenRouteService — documentación de la API. https://openrouteservice.org/dev/#/api-docs